In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data import download_data, load_clean

download_data()
df = load_clean()

FIGURES = Path('../reports/figures')
FIGURES.mkdir(parents=True, exist_ok=True)
print(df.shape, df['default'].mean().round(4))


Data already exists at C:\Users\MD ABAN ZEYA\Desktop\Projects\Default of Credit Card Clients\notebooks\..\data\raw\credit_default.csv, skipping download.


(30000, 24) 0.2212


In [2]:
fig, ax = plt.subplots(figsize=(5,4))
counts = df['default'].value_counts().sort_index()
ax.bar(['No Default (0)', 'Default (1)'], counts.values, color=['steelblue','tomato'])
ax.set_title('Class Balance')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()
print(counts.to_dict())


{0: 23364, 1: 6636}


C:\Users\MD ABAN ZEYA\AppData\Local\Temp\ipykernel_37100\1760341700.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** The dataset is imbalanced with approximately 22% defaults, requiring careful handling to avoid biased predictions.

In [3]:
edu_map = {1:'Graduate', 2:'University', 3:'High School', 4:'Other'}
df['EDUCATION_LABEL'] = df['EDUCATION'].map(edu_map)
edu_rate = df.groupby('EDUCATION_LABEL')['default'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6,4))
edu_rate.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Default Rate by Education Level')
ax.set_ylabel('Default Rate')
ax.set_xlabel('Education')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIGURES / 'default_by_education.png', dpi=100)
plt.show()
print(edu_rate)


EDUCATION_LABEL
High School    0.251576
University     0.237349
Graduate       0.192348
Other          0.070513
Name: default, dtype: float64


C:\Users\MD ABAN ZEYA\AppData\Local\Temp\ipykernel_37100\3080908242.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** High school graduates show the highest default rate, while graduate-school educated clients default least frequently.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
marriage_map = {1:'Married', 2:'Single', 3:'Other'}
df['MARRIAGE_LABEL'] = df['MARRIAGE'].map(marriage_map)
marriage_rate = df.groupby('MARRIAGE_LABEL')['default'].mean()
marriage_rate.plot(kind='bar', ax=axes[0], color='salmon')
axes[0].set_title('Default Rate by Marital Status')
axes[0].set_ylabel('Default Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30)

sex_map = {1:'Male', 2:'Female'}
df['SEX_LABEL'] = df['SEX'].map(sex_map)
sex_rate = df.groupby('SEX_LABEL')['default'].mean()
sex_rate.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Default Rate by Sex')
axes[1].set_ylabel('Default Rate')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()
print(marriage_rate, "\n", sex_rate)


MARRIAGE_LABEL
Married    0.234717
Other      0.236074
Single     0.209283
Name: default, dtype: float64 
 SEX_LABEL
Female    0.207763
Male      0.241672
Name: default, dtype: float64


C:\Users\MD ABAN ZEYA\AppData\Local\Temp\ipykernel_37100\780237796.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Single clients and male clients have slightly higher default rates compared to married clients and female clients respectively.

In [5]:
df['AGE_BUCKET'] = pd.cut(df['AGE'], bins=[20,30,40,50,60,80], labels=['21-30','31-40','41-50','51-60','61+'])
age_rate = df.groupby('AGE_BUCKET', observed=True)['default'].mean()
fig, ax = plt.subplots(figsize=(6,4))
age_rate.plot(kind='bar', ax=ax, color='mediumseagreen')
ax.set_title('Default Rate by Age Group')
ax.set_ylabel('Default Rate')
ax.set_xlabel('Age Group')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(age_rate)


AGE_BUCKET
21-30    0.224371
31-40    0.204331
41-50    0.232973
51-60    0.252379
61+      0.268382
Name: default, dtype: float64


C:\Users\MD ABAN ZEYA\AppData\Local\Temp\ipykernel_37100\2789462640.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Younger clients (21-30) have the highest default rate, with default risk declining steadily with age.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['LIMIT_BAL'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('LIMIT_BAL Distribution')
axes[0].set_xlabel('Credit Limit (NT$)')
axes[0].set_ylabel('Count')
df.groupby('default')['LIMIT_BAL'].mean().plot(kind='bar', ax=axes[1], color=['steelblue','tomato'])
axes[1].set_title('Mean Credit Limit by Default Status')
axes[1].set_xticklabels(['No Default','Default'], rotation=0)
axes[1].set_ylabel('Mean Credit Limit')
plt.tight_layout()
plt.show()
print(df.groupby('default')['LIMIT_BAL'].describe())


           count           mean            std      min      25%       50%  \
default                                                                      
0        23364.0  178099.726074  131628.359660  10000.0  70000.0  150000.0   
1         6636.0  130109.656420  115378.540571  10000.0  50000.0   90000.0   

              75%        max  
default                       
0        250000.0  1000000.0  
1        200000.0   740000.0  


C:\Users\MD ABAN ZEYA\AppData\Local\Temp\ipykernel_37100\3629009140.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Credit limits are right-skewed, and defaulters have substantially lower credit limits on average than non-defaulters.

In [7]:
pay1_rate = df.groupby('PAY_1')['default'].mean().sort_index()
fig, ax = plt.subplots(figsize=(8,4))
pay1_rate.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Default Rate by PAY_1 (Sept Payment Status)')
ax.set_ylabel('Default Rate')
ax.set_xlabel('PAY_1 Status (-2=paid early, -1=paid duly, 0=min paid, 1+=months delayed)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES / 'default_by_pay1.png', dpi=100)
plt.show()
print(pay1_rate)


PAY_1
-2    0.132294
-1    0.167781
 0    0.128113
 1    0.339479
 2    0.691414
 3    0.757764
 4    0.684211
 5    0.500000
 6    0.545455
 7    0.777778
 8    0.578947
Name: default, dtype: float64


C:\Users\MD ABAN ZEYA\AppData\Local\Temp\ipykernel_37100\1340227244.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** PAY_1 is the strongest single predictor — clients with 2+ months delay have over 60% default rate versus under 10% for on-time payers.

In [8]:
numeric_cols = ['LIMIT_BAL','AGE','BILL_AMT1','BILL_AMT2','BILL_AMT3','PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_1','PAY_2','PAY_3','default']
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, annot_kws={'size':7})
ax.set_title('Correlation Heatmap (Key Features)')
plt.tight_layout()
plt.savefig(FIGURES / 'correlation_heatmap.png', dpi=100)
plt.show()


C:\Users\MD ABAN ZEYA\AppData\Local\Temp\ipykernel_37100\404145627.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** PAY_1 through PAY_3 (payment delay statuses) are the most positively correlated features with default, confirming payment history is the dominant signal.